# Kütüphaneler

In [28]:
import pandas as pd
import numpy as np
import pandas.io.formats.excel as pife
import warnings
import pickle
import os
import locale
from termcolor import colored
from pandas.errors import SettingWithCopyWarning
from datetime import datetime

# Ayarlar

In [29]:
os.system('color')
locale.setlocale(locale.LC_ALL, 'tr_TR')
pd.options.display.float_format = '{:,.2f}'.format
pife.ExcelFormatter.header_style = None

warnings.filterwarnings('ignore', category = UserWarning)
warnings.filterwarnings('ignore', category = pd.errors.ParserWarning)
warnings.filterwarnings('ignore', category = SettingWithCopyWarning)

base_path = 'C:\\Users\\105617\\Desktop\\Workspace\\Panel\\'

pickle_folder_path = base_path + 'pickle' + '\\'
data_folder_path = base_path + 'data' + '\\'
output_folder_path = base_path + 'output' + '\\'

# Yardımcı Fonksiyonlar

## Tablo Manipülasyonu

In [30]:
def add_h_space(dfo, space=1):
    df = dfo.copy()
    w = df.shape[1]
    for _ in range(space):
        df.loc[len(df)] = [np.nan] * w
    return df

def add_v_space(dfo, space=1):
    df = dfo.copy()
    return pd.concat([df, pd.DataFrame([[np.nan] * space], columns=[np.nan] * space)], axis=1)
    

def h_stack(dfo, space=3):
    df = dfo.copy()
    df = df[0].reset_index(drop=True)
    dfln = [df]
    dfna = pd.DataFrame({np.nan: [np.nan]})
    for dftt in dfo[1:]:
        dft = dftt.copy()
        for i in range(space):
            dfln.append(dfna)
        dfln.append(dft.reset_index(drop=True))
    df = pd.concat(dfln, axis=1).reset_index(drop=True)
    return df


def v_stack(dfo, space=3):
    df = dfo.copy()
    df = df[0].reset_index(drop=True)
    dfln = [df]
    dfna = pd.DataFrame(columns=df.columns)

    dfna = add_h_space(dfna, space)
    for dftt in dfo[1:]:
        dft = dftt.copy()
        l = df.shape[1]
        lt = dft.shape[1]
        diff = l - lt

        if diff > 0:
            dft = add_v_space(dft, diff)
        elif diff < 0:
            df = add_v_space(df, -diff)

        dfc = pd.DataFrame([dft.columns], columns=df.columns)
        dft.columns = df.columns
        dfln.append(dfna)
        dfln.append(dfc)
        dfln.append(dft.reset_index(drop=True))

    df = pd.concat(dfln).reset_index(drop=True)
    return df


def insert_header(dfo, col):
    df = dfo.copy()
    header = [col] if type(col) is str else col
    return v_stack([pd.DataFrame(columns=header), df], 0)

def insert_corner(dfo):
    df = dfo.copy()
    return h_stack([pd.DataFrame(), v_stack([pd.DataFrame(), df], 0)], 1)

def insert_row(dfo, index=0, row=None, count=1):
    df = dfo.copy().reset_index(drop=True)
    if row is None:
        row = np.nan
    if type(row) is list and len(row) < df.shape[1]:
        row += [np.nan] * (df.shape[1] - len(row))
    if index < 0:
        index += df.shape[0]
    new_indices = [x for x in range(index)] + [x + count for x in range(index, len(df))]
    df.index = new_indices
    for i in range(count):
        df.loc[index + i] = row
    df = df.reset_index().sort_values('index').drop('index', axis=1).reset_index(drop=True)
    return df

## Verim Tabloları

In [31]:
def fix_verim_table(df, columns=None):
    start = 0
    if columns is None:
        start = 3
        columns = df.iloc[2, 1:]
    df = df.iloc[start:, 1:]
    df.columns = columns
    df.columns.name = None
    df = df.reset_index(drop=True)
    return df

def quick_export_verim_csv(data, output_file_name, sep=';', header=False, index=False, open_file=False):
    cl = [str(x) for x in range(0, 16)]
    df = pd.DataFrame()
    df['10'] = data
    for c in ['0', '15']:
        df[c] = 'x'
    for c in cl:
        if c not in ['0', '10', '15']:
            df[c] = np.nan
    df = df[cl]

    output_file_path = output_folder_path + output_file_name + '.csv'
    df.to_csv(output_file_path, sep=sep, header=header, index=index)

    if open_file:
        os.startfile(output_file_path)
    

## Hızlı Çıktı

In [32]:
def quick_export(df_export, output_file_name, sheet_name=None, open_file=True):
    output_file_path = output_folder_path + output_file_name + '.xlsx'
    writer = pd.ExcelWriter(output_file_path, engine = 'xlsxwriter')
    if type(df_export) is list:
        if sheet_name is None:
            sheet_name = [x + 1 for x in range(len(df_export))]
        for df, sheet in zip(df_export, sheet_name):
            df.to_excel(writer, sheet_name = sheet, index = False)
    else:
        if sheet_name is None:
            sheet_name = output_file_name
        df_export.to_excel(writer, sheet_name = sheet_name, index = False)
    writer.close()
    if open_file:
        os.startfile(output_file_path)


def quick_export_csv(df, file_name, open_file=False):
    df.to_csv(output_folder_path + file_name  + '.csv', sep=';', encoding='ANSI', index=False)

## Now

In [33]:
def now():
    # return datetime.strftime(datetime.now(), '%Y-%m-%d_%H-%M-%S')
    return datetime.strftime(datetime.now(), '%m%d%H%M')

## Pickle

In [34]:
def save_pickle(file, name, sub=''):
    os.makedirs(pickle_folder_path + sub, exist_ok = True)
    with open(pickle_folder_path + sub + name, 'wb') as handle:
        pickle.dump(file, handle, protocol=pickle.HIGHEST_PROTOCOL)

def load_pickle(name, sub=''):
    with open(pickle_folder_path + sub + name, 'rb') as handle:
        return pickle.load(handle)

# Listeler

In [73]:
segment_list, segment_title_list, segment_title_dict, bolum_list, filtered_bolum_list, sektor_list, filtered_sektor_list, sorted_sektor_list, sektor_title_dict, sektor_title_short_dict, filtered_sube_turu_list, sube_list, bolge_list = load_pickle('list_list')

# Validasyon

In [36]:
# start_term, end_term = '2022.12', '2023.12'
# last_date = end_term
# pra_v = 'v11'


# df_wide = load_pickle('df_ms_all')
# sorted_date_list = load_pickle('sorted_date_list_v_2312')
# df_new = load_pickle(f'df_pra_{pra_v}_2312')
# df_old = load_pickle(f'df_pra_{pra_v}_2212')




# df_wide = load_pickle('df_wide_v_2312')
# df_wide = load_pickle('df_ms_all_2311')
# sorted_date_list = load_pickle('sorted_date_list_v_2311')
# df_new = load_pickle(f'df_pra_{pra_v}_2311')
# df_old = load_pickle(f'df_pra_{pra_v}_2212')

In [108]:
pra_v, start_term, end_term = 'v11', '2022.12', '2023.12'

last_date = end_term
old_suffix = f'{start_term[2:4]}{start_term[-2:]}'
new_suffix = f'{end_term[2:4]}{end_term[-2:]}'

df_wide = load_pickle(f'df_ms_all_{new_suffix}')
sorted_date_list = load_pickle(f'sorted_date_list_v_{new_suffix}')
df_new = load_pickle(f'df_pra_{pra_v}_{new_suffix}')
df_old = load_pickle(f'df_pra_{pra_v}_{old_suffix}')

## Max MS

In [109]:
def find_max_column(df, columns):
    values = [df[c] for c in columns if not np.isnan(df[c])]
    if len(values): return max(values)
    else: return np.nan
    
def find_first_occurrence(df, columns, threshold):  
    return list(df[columns].values).index(threshold) + 1

def get_df_pra_max(df_pra, date_list, column='Müşteri Sınıfı'):
    c = column
    sorted_date_list = date_list
    mn = 'Müşteri No'
    a = 'Adı'

    df = df_pra.copy()
    si = sorted_date_list.index(start_term) + 1
    ei = sorted_date_list.index(end_term) + 1
    dfl = df_wide[si:ei]
    dl = sorted_date_list[si:ei]

    for dft, d in zip(dfl, dl):
        df = pd.merge(df, dft[['Müşteri No', 'Müşteri Sınıfı ' + d]], on=mn, how='left')

    cs = [c + ' ' + d for d in dl]
    df[c + ' Max'] = df.apply(find_max_column, columns=cs, axis=1)
    df['2. Sınıfa Geçiş Ayı'] = np.nan
    df['3. Sınıfa Geçiş Ayı'] = np.nan

    df.loc[df[c + ' Max'] == 2, '2. Sınıfa Geçiş Ayı'] = df.loc[df[c + ' Max'] == 2].apply(find_first_occurrence, columns=cs, threshold=2, axis=1)
    df.loc[df[c + ' Max'] == 3, '3. Sınıfa Geçiş Ayı'] = df.loc[df[c + ' Max'] == 3].apply(find_first_occurrence, columns=cs, threshold=3, axis=1)

    return df.copy()

## Geçiş Özet

In [110]:
def create_pra_pass(df_old_max, last_date=last_date):
    rk_list = ['Mükemmel', 'Çok Düşük', 'Düşük', 'Orta', 'Yüksek', 'Çok Yüksek']
    rk = 'Risk Kategorisi'
    mn = 'Müşteri No'
    nrt = 'Nakdi Reeskontlu Toplam'
    msm = 'Müşteri Sınıfı Max'
    msl = 'Müşteri Sınıfı ' + last_date
    sga2 = '2. Sınıfa Geçiş Ayı'
    sga3 = '3. Sınıfa Geçiş Ayı'
    factor = 1_000_000
    dfv_list = []


    for find_risk, x, s in zip([False, True], [mn, nrt], ['Adet', 'Nakdi Risk']):
        dfp = df_old_max.copy()
        if find_risk:
            dfp[nrt] /= factor

        t_list = ['Genel Toplam']
        df = pd.DataFrame()
        df[rk] = rk_list

        c = x
        n = s
        dft = dfp[[rk, c]].copy()
        if find_risk:
            dft = dft.groupby(rk).sum().reset_index().rename(columns={c: n})
        else:
            dft = dft.groupby(rk).count().reset_index().rename(columns={c: n})
        df = pd.merge(df, dft, on=rk, how='left')
        t = df[n].sum()
        t_list.append(t)

        df = add_v_space(df)
        t_list.append(np.nan)

        for ss in [2, 3]:
            c = x
            n = f'{ss}. Sınıfa Geçen Adet'
            ga = f'{ss}. Sınıfa Geçiş Ayı'
            dfa = dfp.loc[dfp[msm] == ss, [rk, c, ga]].copy()
            if find_risk:
                dft = dfa[[rk, c]].groupby(rk).sum().reset_index().rename(columns={c: n})
            else:
                dft = dfa[[rk, c]].groupby(rk).count().reset_index().rename(columns={c: n})
            df = pd.merge(df, dft, on=rk, how='left')
            t = df[n].sum()
            t_list.append(t)

            p = f'{ss}. Sınıfa Geçiş %'
            df[p] = df[n] / df[s]
            t = df[n].sum() / df[s].sum()
            t_list.append(t)

            c = f'{ss}. Sınıfa Geçiş Ayı'
            n = f'Ortalama {ss}. Sınıfa Geçiş Ayı'
            t = dfa[ga].mean()
            t_list.append(t)
            dft = dfa[[rk, ga]].groupby(rk).mean().reset_index().rename(columns={c: n})
            df = pd.merge(df, dft, on=rk, how='left')

            df = add_v_space(df)
            t_list.append(np.nan)

        c = x
        n = 'Riski Kapanan'
        dfa = dfp.loc[dfp[msl].isnull(), [rk, c]].copy()
        if find_risk:
            dft = dfa.groupby(rk).sum().reset_index().rename(columns={c: n})
        else:
            dft = dfa.groupby(rk).count().reset_index().rename(columns={c: n})
        df = pd.merge(df, dft, on=rk, how='left')
        t = df[n].sum()
        t_list.append(t)

        p = 'Riski Kapanma %'
        df[p] = df[n] / df[s]
        t = df[n].sum() / df[s].sum()
        t_list.append(t)

        df = add_h_space(df)
        df.loc[len(df)] = t_list

        header = 'Nakdi Riske Göre' if find_risk else 'Adede Göre'
        df = insert_header(df, header)
        footer_end = 'Reeskontlu, mio TL' if find_risk else 'Adet'
        footer = f'{start_term}-{end_term} KR202, {footer_end}'
        df.loc[len(df)] = [footer] + [np.nan] * (len(df.columns) - 1)
        df.loc[len(df)] = ['Riski kapanan firmalar dahildir'] + [np.nan] * (len(df.columns) - 1)
        
        dfv_list.append(df.copy())

    return insert_corner(v_stack(dfv_list))


## Model Karşılaştırma

In [111]:
def create_pra_model_comp(df_pra_max):
    dfp = df_pra_max.copy()
    rk_list = ['Düşük Riskli', 'Yüksek Riskli']
    rk_upper = ['Yüksek', 'Çok Yüksek']
    rk = 'Risk Kategorisi'
    mn = 'Müşteri No'
    nrt = 'Nakdi Reeskontlu Toplam'
    msm = 'Müşteri Sınıfı Max'
    factor = 1_000_000

    dfv_list_list_list = []
    for find_risk, c in zip([False, True], [mn, nrt]):
        dfv_list_list = []
        for ss in [2, 3]:
            dfv_list = []
            for x, y, th in zip(['Entegre Derece Skor', 'Yapay Zeka Risk Sınıfı'], ['Entegre Derece / Skor', 'Yapay Zeka Risk Sınıfı'], [[(1, 7), (8, 12)], [(1, 2), (3, 5)]]):
                dfa = dfp.loc[dfp[msm] == ss, [rk, c, x]]
                df = pd.DataFrame()
                df[rk] = rk_list
                df = add_v_space(df)
                t_list = ['Genel Toplam', np.nan]
                for lower, upper in th:
                    n = f'{lower} - {upper}'
                    dft = dfa.loc[(dfa[x] >= lower) & (dfa[x] <= upper), [rk, c]]
                    if find_risk:
                        u = dft.loc[dfp[rk].isin(rk_upper), c].sum()
                        t = dft[c].sum()
                    else:
                        u = dft.loc[dfp[rk].isin(rk_upper), c].count()
                        t = dft[c].count()
                    t_list.append(t)
                    l = t - u
                    df[n] = [l, u]

                
                df = add_v_space(df)
                t_list.append(np.nan)

                n = 'Genel Toplam'
                if find_risk:
                    u = dfa.loc[dfa[rk].isin(rk_upper), c].sum()
                    t = dfa[c].sum()
                else:
                    u = dfa.loc[dfa[rk].isin(rk_upper), c].count()
                    t = dfa[c].count()
                t_list.append(t)
                l = t - u
                df[n] = [l, u]
                
                df = add_h_space(df)
                df.loc[len(df)] = t_list
                if find_risk:
                    df.iloc[:, 1:] /= factor
                    

                header_end = 'Nakdi Risk' if find_risk else 'Müşteri Adedi'
                header = f'{ss}. Sınıfa Geçen {header_end}'
                dfx = pd.DataFrame(columns=[header, np.nan, y])
                df = v_stack([dfx, df], 0)
                footer_end = 'Reeskontlu, mio TL' if find_risk else 'Adet'
                footer = f'{start_term}-{end_term} KR202, {footer_end}'
                df.loc[len(df)] = [footer] + [np.nan] * (len(df.columns) - 1)
                df.loc[len(df)] = ['Riski kapanan firmalar dahildir'] + [np.nan] * (len(df.columns) - 1)

                dfv_list.append(df)
            dfv_list_list.append(h_stack(dfv_list, 4))
        dfv_list_list_list.append(v_stack(dfv_list_list))

    return insert_corner(v_stack(dfv_list_list_list, 6))

## Değişim

### Alt

In [126]:
def create_pra_dist_alt(df_pra, rows, row_list, use_thresholds=False, reorder=False, rename_dict=None, factor=1_000_000, find_risk=False):
    n_list = ['Mükemmel', 'Çok Düşük', 'Düşük', 'Orta', 'Yüksek', 'Çok Yüksek']
    rk = 'Risk Kategorisi'
    yrc = 'Yüksek Riskli Portföy Oranı'
    mn = 'Müşteri No'
    nrt = 'Nakdi Reeskontlu Toplam'
    c = rows
    c_list = row_list
    dfp = df_pra.copy()
    x = nrt if find_risk else mn

    df = pd.DataFrame()
    if use_thresholds:
        ettl = c_list
        etl = [f'{et[0]} - {et[1]}' for et in ettl]
        df[c] = etl
    else:
        df[c] = c_list if c_list is not None else sorted(df_pra.loc[df_pra[c].notnull(), c].unique())
        
    t_list = ['Genel Toplam']

    for n in n_list:
        dft = dfp.copy()
        if find_risk:
            dft = dft.loc[dft[rk] == n, [x, c]].groupby(c).sum().reset_index().rename(columns={x: n})
        else:
            dft = dft.loc[dft[rk] == n, [x, c]].groupby(c).count().reset_index().rename(columns={x: n})
        if use_thresholds:
            nx_list = []
            for et in ettl:
                nx = dft.loc[(dft[c] >= et[0]) & (dft[c] <= et[1]), n].sum()
                nx_list.append(nx)
            df[n] = nx_list
        else:
            df = pd.merge(df, dft, on=c, how='outer')
        t_list.append(df[n].sum())

    n = 'Toplam'
    dft = dfp.copy()
    if find_risk:
        dft = dft[[x, c]].groupby(c).sum().reset_index().rename(columns={x: n})
    else:
        dft = dft[[x, c]].groupby(c).count().reset_index().rename(columns={x: n})
    if use_thresholds:
        nx_list = []
        for et in ettl:
            nx = dft.loc[(dft[c] >= et[0]) & (dft[c] <= et[1]), n].sum()
            nx_list.append(nx)
        df[n] = nx_list
    else:
        df = pd.merge(df, dft, on=c, how='outer')
    t_list.append(df[n].sum())

    df = df.dropna(axis=0, how='all', subset=list(df.columns)[1:], ignore_index=True)
    if rename_dict is not None:
        df[c] = df[c].map(rename_dict)
    df = add_h_space(df)
    df.loc[len(df)] = t_list
    if find_risk:
        df.iloc[:,1:] = df.iloc[:,1:] / factor
    df[yrc] = df[['Yüksek', 'Çok Yüksek']].sum(axis=1) / df['Toplam']
    if reorder:
        df.iloc[:-1, :] = df.iloc[:-1, :].sort_values(yrc, ascending=False)


    return df[[c, yrc]]

### Değişim

In [127]:
def create_pra_dist_change(df_old, df_new, rows, row_list, row_new=None, sort=0, use_thresholds=False, drop_na=True, study_scope=None):
    dfn = df_new.copy()
    dfo = df_old.copy()

    yrpo = 'Yüksek Riskli Portföy Oranı'

    r = rows
    r_list = row_list
    r_dict = row_new
    
    dfv_list = []

    for find_risk in [False, True]:
        df = pd.DataFrame()
        if use_thresholds:
            df[r] = [f'{et[0]} - {et[1]}' for et in r_list] + ['Genel Toplam']
        else:
            df[r] = r_list + ['Genel Toplam']
        df = add_v_space(df)

        dft = create_pra_dist_alt(dfo, r, r_list, use_thresholds, find_risk=find_risk)
        dft.columns = [r, start_term]
        df = pd.merge(df, dft, on=r, how='left')
        dft = create_pra_dist_alt(dfn, r, r_list, use_thresholds, find_risk=find_risk)
        dft.columns = [r, end_term]
        df = pd.merge(df, dft, on=r, how='left')

        if drop_na:
            df = df.dropna(subset=list(df.columns)[2:], how='all')

        df['Değişim'] = df[end_term] - df[start_term]

        if r_dict is not None:
            if type(r_dict) is list:
                r_dict = dict(zip(r_list, r_dict))
            df[r] = df[r].apply(lambda x: r_dict.get(x, x))

        if sort is not None:
            df.loc[df[end_term] == 0, 'Değişim'] = df.loc[df[end_term] == 0, start_term] * -100
            df.loc[(df[end_term] == 0) & (df[start_term] == 0), 'Değişim'] = -1_000_000
            if type(sort) is str:
                if sort.startswith('-'):
                    df[:-1] = df[:-1].sort_values(sort[1:], ascending=False)
                else:
                    df[:-1] = df[:-1].sort_values(sort)
            else:
                if sort > 0:
                    df[:-1] = df[:-1].sort_values('Değişim')
                elif sort < 0:
                    df[:-1] = df[:-1].sort_values('Değişim', ascending=False)
            df['Değişim'] = df[end_term] - df[start_term]

        df = insert_row(df, -1)

        header = 'Nakdi Riske Göre' if find_risk else 'Adede Göre'
        df = insert_header(df, header)
        df = insert_row(df, 0, [np.nan, np.nan, yrpo])
        footer_end = 'Reeskontlu' if find_risk else 'Adet'
        footer = f'{start_term}-{end_term} KR202, {footer_end}'
        df.loc[len(df)] = [footer] + [np.nan] * (len(df.columns) - 1)
        
        if study_scope is not None:
            footer_scope = f'{start_term} ve {end_term} çalışma kümelerine {study_scope[0]} ve üzeri müşterisi giren {study_scope[1]} dikkate alınmıştır'
            df.loc[len(df)] = [footer_scope] + [np.nan] * (len(df.columns) - 1)
            df = add_h_space(df)

        dfv_list.append(df.copy())

    return insert_corner(v_stack(dfv_list))

### Scope

In [138]:
def create_pra_dist_change_scope(df_old, df_new, rows, row_list, row_new=None, use_thresholds=False, scope=20, study_scope=(30, 'şubeler')):
    dfn = df_new.copy()
    dfo = df_old.copy()

    yrpo = 'Yüksek Riskli Portföy Oranı'
    r = rows
    r_list = row_list
    r_dict = row_new

    dfv_list = []

    for find_risk in [False, True]:
        df = pd.DataFrame()
        if use_thresholds:
            df[r] = [f'{et[0]} - {et[1]}' for et in r_list] + ['Genel Toplam']
        else:
            df[r] = r_list + ['Genel Toplam']
        df = add_v_space(df)

        dft = create_pra_dist_alt(dfo, r, r_list, use_thresholds, find_risk=find_risk)
        dft.columns = [r, start_term]
        df = pd.merge(df, dft, on=r, how='left')
        dft = create_pra_dist_alt(dfn, r, r_list, use_thresholds, find_risk=find_risk)
        dft.columns = [r, end_term]
        df = pd.merge(df, dft, on=r, how='left')
        df['Değişim'] = df[end_term] - df[start_term]

        if r_dict is not None:
            if type(r_dict) is list:
                r_dict = dict(zip(r_list, r_dict))
            df[r] = df[r].apply(lambda x: r_dict.get(x, x))

        df.loc[df[end_term] == 0, 'Değişim'] = df.loc[df[end_term] == 0, start_term] * -100
        df.loc[(df[end_term] == 0) & (df[start_term] == 0), 'Değişim'] = -1_000_000
        
        dfg = df.copy()
        dfg[:-1] = dfg[:-1].sort_values('Değişim')
        dfg = pd.concat([dfg[:scope], dfg[-1:]]).reset_index(drop=True)

        dfb = df.copy()
        dfb[:-1] = dfb[:-1].sort_values('Değişim', ascending=False)
        dfb = pd.concat([dfb[:scope], dfb[-1:]]).reset_index(drop=True)

        dfg['Değişim'] = dfg[end_term] - dfg[start_term]
        dfb['Değişim'] = dfb[end_term] - dfb[start_term]
        dfg = insert_row(dfg, -1)
        dfb = insert_row(dfb, -1)

        header_end = 'Nakdi Riske Göre' if find_risk else 'Adede Göre'
        footer_end = 'Reeskontlu' if find_risk else 'Adet'
        footer = f'{start_term}-{end_term} KR202, {footer_end}'
        header_good = f'İyileşen {scope}, {header_end}'
        header_bad = f'Kötüleşen {scope}, {header_end}'
        footer = f'{start_term}-{end_term} KR202, {footer_end}'
        footer_scope = f'{start_term} ve {end_term} çalışma kümelerine {study_scope[0]} ve üzeri müşterisi giren {study_scope[1]} dikkate alınmıştır'

        dfg = insert_header(dfg, header_good)
        dfg = insert_row(dfg, 0, [np.nan, np.nan, yrpo])
        dfg.loc[len(dfg)] = [footer] + [np.nan] * (len(dfg.columns) - 1)
        dfg.loc[len(dfg)] = [footer_scope] + [np.nan] * (len(dfg.columns) - 1)
        dfg = add_h_space(dfg)

        dfb = insert_header(dfb, header_bad)
        dfb = insert_row(dfb, 0, [np.nan, np.nan, yrpo])
        dfb.loc[len(dfb)] = [footer] + [np.nan] * (len(dfb.columns) - 1)
        dfb.loc[len(dfb)] = [footer_scope] + [np.nan] * (len(dfb.columns) - 1)
        dfb = add_h_space(dfb)

        dfv_list.append(h_stack([dfb, dfg]))

    return insert_corner(v_stack(dfv_list))

### Adet-Risk Değişimi

In [139]:
def create_pra_count_risk_change(df_old, df_new, rows, row_list, row_new=None, drop_na=True, sort=True, factor=1_000_000):

    dfn = df_new.copy()
    dfo = df_old.copy()

    dfn = dfn.loc[dfn['Risk Kategorisi'].isin(['Çok Yüksek', 'Yüksek'])]
    dfo = dfo.loc[dfo['Risk Kategorisi'].isin(['Çok Yüksek', 'Yüksek'])]
    r = rows
    r_list = row_list
    r_dict = row_new

    dfv_list = []

    for c, nn, pp, find_risk in zip(['Müşteri No', 'Nakdi Reeskontlu Toplam'], ['Adet', 'Nakdi Risk'], ['Adet Payı', 'Nakdi Risk Payı'], [False, True]):
        df = pd.DataFrame()
        df[r] = r_list
        df = add_v_space(df)
        t_list = ['Genel Toplam', np.nan]
        start = start_term + ' ' + pp
        end = end_term + ' ' + pp

        for dfp, d in zip([dfo, dfn], [start_term, end_term]):
            n = d + ' ' + nn
            p = d + ' ' + pp
            dft = dfp[[r, c]].copy()
            if find_risk:
                dft = dft.groupby(r).sum().reset_index().rename(columns={c: n})
                dft[n] /= factor
            else:
                dft = dft.groupby(r).count().reset_index().rename(columns={c: n})
            df = pd.merge(df, dft, on=r, how='outer')
            t = df[n].sum()
            t_list.append(t)
            df[p] = df[n] / t
            t_list.append(1)

        if drop_na:
            df = df.dropna(subset=df.columns[2:], how='all').reset_index(drop=True)

        if r_dict is not None:
            df[r] = df[r].apply(lambda x: r_dict.get(x, x))

        if sort:
            df = df.sort_values([start, end, r], ascending=False).reset_index(drop=True)

        df.iloc[:, 2:] = df.iloc[:, 2:].fillna(0)

        df = add_h_space(df)
        df.loc[len(df)] = t_list

        df['Pay Değişimi'] = df[end] - df[start]
        df.iloc[-1, -1] = np.nan


        header = 'Nakdi Riske Göre' if find_risk else 'Adede Göre'
        df = insert_header(df, 'Yüksek Riskli Portföy İçerisinde')
        df = insert_header(df, header)

        footer_end = 'Reeskontlu, mio TL' if find_risk else 'Adet'
        footer = f'{start_term}-{end_term} KR202, {footer_end}'
        df.loc[len(df)] = [footer] + [np.nan] * (len(df.columns) - 1)

        dfv_list.append(df.copy())

    return insert_corner(v_stack(dfv_list))

## Filtrelenmiş Liste

In [140]:
def get_scope_filtered_list(df_old, df_new, column, filter_value=None, filter_column=None, scope=30):
    dfo = df_old.copy()
    dfn = df_new.copy()
    c = column

    if filter_value is not None:
        x = c if filter_column is None else filter_column
        y = filter_value
        dfo = dfo.loc[dfo[x] == y]
        dfn = dfn.loc[dfn[x] == y]

    dfo = dfo[c].value_counts().reset_index().rename(columns={'count': 'old'})
    dfn = dfn[c].value_counts().reset_index().rename(columns={'count': 'new'})
    df = pd.merge(dfo, dfn, on=c, how='inner')
    filtered_list = list(df.loc[df.apply(lambda x: x['old'] >= scope and x['new'] >= scope, axis=1), c])
    filtered_list.sort(key=locale.strxfrm)

    return filtered_list

# Analiz

In [141]:
df_pra_old_max = get_df_pra_max(df_old, sorted_date_list)
df_pra_gecis = create_pra_pass(df_pra_old_max)
df_pra_model = create_pra_model_comp(df_pra_old_max)

In [142]:
df_pra_segment_change = create_pra_dist_change(df_old, df_new, 'Değerlendirmeye Esas Segment', segment_list, segment_title_list)
df_pra_bolum_change = create_pra_dist_change(df_old, df_new, 'İlgili Bölüm', filtered_bolum_list)
df_pra_sube_change = create_pra_dist_change(df_old, df_new, 'Şube Türü', filtered_sube_turu_list)
df_pra_eds_change = create_pra_dist_change(df_old, df_new, 'Entegre Derece Skor', [(1, 5), (6, 7), (8, 12)], use_thresholds=True)
df_pra_bolge_change = create_pra_dist_change(df_old, df_new, 'Bölge Adı', bolge_list, sort=-1)

In [143]:
df_pra_sektor_count_risk_change = create_pra_count_risk_change(df_old, df_new, 'Tahsis Sektör Adı', sorted_sektor_list, sektor_title_dict)

sektor_scope_list = get_scope_filtered_list(df_old, df_new, 'Tahsis Sektör Adı', scope=1000)
df_pra_sektor_change = create_pra_dist_change(df_old, df_new, 'Tahsis Sektör Adı', sorted_sektor_list, sektor_title_dict)
df_pra_sektor_scope_change = create_pra_dist_change(df_old, df_new, 'Tahsis Sektör Adı', sektor_scope_list, sektor_title_dict, '-'+start_term, study_scope=(1000, 'sektörler'))

In [144]:
kurumsal_sube_scope_list = get_scope_filtered_list(df_old, df_new, 'Şube Adı', 'Kurumsal', 'Şube Türü', 5)
ticari_sube_scope_list = get_scope_filtered_list(df_old, df_new, 'Şube Adı', 'Ticari', 'Şube Türü')
karma_sube_scope_list = get_scope_filtered_list(df_old, df_new, 'Şube Adı', 'Karma', 'Şube Türü')

df_pra_kurumsal_sube_change = create_pra_dist_change(df_old, df_new, 'Şube Adı', kurumsal_sube_scope_list, sort=1)
df_pra_ticari_sube_change = create_pra_dist_change_scope(df_old, df_new, 'Şube Adı', ticari_sube_scope_list)
df_pra_karma_sube_change = create_pra_dist_change_scope(df_old, df_new, 'Şube Adı', karma_sube_scope_list)

In [145]:
# quick_export(df_pra_old_max, f'PRA Tüm Firmalar {start_term}', f'Tüm Firmalar {start_term}', False)
# quick_export(df_new, f'PRA Tüm Firmalar {end_term}', f'Tüm Firmalar {end_term}', False)

# Başlıklar

In [146]:
sheet_dict = {
    'Geçişler': df_pra_gecis,
    'Modeller': df_pra_model,
    'Sektör Dağılım': df_pra_sektor_count_risk_change,
    'Sektör Tüm': df_pra_sektor_change,
    'Sektör': df_pra_sektor_scope_change,
    'Segment': df_pra_segment_change,
    'Bölüm': df_pra_bolum_change,
    'EDS': df_pra_eds_change,
    'Bölge': df_pra_bolge_change,
    'Şube': df_pra_sube_change,
    'Kurumsal': df_pra_kurumsal_sube_change,
    'Ticari': df_pra_ticari_sube_change,
    'Karma': df_pra_karma_sube_change,
}

eds = 'Entegre Derece / Skor'
yzrs = 'Yapay Zeka Risk Sınıfı'
yrpo = 'Yüksek Riskli Portföy Oranı'

merge_dict = {
    'Modeller': {eds: ['D2:E2', 'D13:E13', 'D27:E27', 'D38:E38'], yzrs: ['N2:O2', 'N13:O13', 'N27:O27', 'N38:O38']},
    'Sektör Tüm': {yrpo: ['D3:F3', 'D45:F45']},
    'Sektör': {yrpo: ['D3:F3', 'D28:F28']},
    'Segment': {yrpo: ['D3:F3', 'D16:F16']},
    'Bölüm': {yrpo: ['D3:F3', 'D15:F15']},
    'EDS': {yrpo: ['D3:F3', 'D15:F15']},
    'Bölge': {yrpo: ['D3:F3', 'D26:F26']},
    'Şube': {yrpo: ['D3:F3', 'D15:F15']},
    'Kurumsal': {yrpo: ['D3:F3', 'D23:F23']},
    'Ticari': {yrpo: ['D3:F3', 'D33:F33', 'L3:N3', 'L33:N33']},
    'Karma': {yrpo: ['D3:F3', 'D33:F33', 'L3:N3', 'L33:N33']},
}


sheet_dict_list = [sheet_dict]

sheet_list = [item for sublist in [list(x.keys()) for x in sheet_dict_list] for item in sublist]
df_list = [item for sublist in [list(x.values()) for x in sheet_dict_list] for item in sublist]
# sheet_list = list(range(len(page_list)))

# sheet_list_list = []
# ll = [len(x.keys()) for x in sheet_dict_list]
# c = 0
# for i, l in enumerate(ll):
#     sheet_list_list.append(list(range(c, c + l)))
#     c += l

# pra_sheet_list = sheet_list_list


# df_index = pd.DataFrame()
# sheet_list = [x for x in range(len(page_list))]
# df_index['Sayfa'] = sheet_list
# df_index['Başlık'] = df_index['Sayfa'].apply(lambda x: '=HYPERLINK("#{}!A1", "{}")'.format(str(x), page_list[x]))
# sheet_color_list = ['E5E1E6', 'C7CEEA', 'B2CBF7', 'B5EAD7', 'E2F0CB', 'F7ECC8', 'FFDAC1', 'FFB7B2', 'FF9AA2', 'DCB5CB', 'AF8FC1']

# Final XLSX

In [147]:
output_file_name = f'PRA {pra_v} Val {start_term}-{end_term}'
open_file = True

color_long_multi_trend, color_multi_trend, color_trend, row_based = True, True, True, True
output_file_path = output_folder_path + output_file_name + '.xlsx'
writer = pd.ExcelWriter(output_file_path, engine = 'xlsxwriter')

workbook = writer.book
# bold_format = workbook.add_format({'bold': True, 'text_wrap': True})
# row_format = workbook.add_format({'align': 'left'})
# link_format = workbook.add_format({'underline': True, 'color': 'blue'})
# float_format = workbook.add_format()
# float_format.set_num_format('#,##0.00')
# int_format = workbook.add_format()
# int_format.set_num_format('#,##0')

# color_format_list = []
# for i in range(len(sheet_list_list)):
#     color = sheet_color_list[i]
#     cf = workbook.add_format()
#     cf.set_bg_color(color)
#     color_format_list.append(cf)

# colored_title_format = workbook.add_format({'bold': True})
# colored_title_format.set_bg_color('C9C4CF')


# df_index.to_excel(writer, sheet_name = 'Başlıklar', index = False)
for sheet, df in zip(sheet_list, df_list):
    # sheet = str(i)
    df.to_excel(writer, sheet_name = sheet, index = False)

    worksheet = writer.sheets[sheet]
    worksheet.hide_gridlines(2)

    if sheet in merge_dict:
        for cell_value in merge_dict[sheet]:
            for cell_range in merge_dict[sheet][cell_value]:
                worksheet.merge_range(cell_range, cell_value)
        


    # for j, c in enumerate(df.columns):
    #     worksheet.set_row(0, 60, bold_format)
    #     worksheet.set_column(0, 0, 18, row_format)
    #     if any(x in str(c).lower() for x in ['%', 'ortalama', '2022', '2023', 'n-']):
    #         worksheet.set_column(j, j, 10, float_format)
    #     else:
    #         worksheet.set_column(j, j, 16, int_format)


# worksheet = writer.sheets['Başlıklar']
# worksheet.set_row(0, None, bold_format)
# worksheet.set_column(1, 1, 70, link_format)
# worksheet.write('A1', df_index.columns[0], colored_title_format)
# worksheet.write('B1', df_index.columns[1], colored_title_format)

# for x, cf in enumerate(color_format_list):
#     for i in sheet_list_list[x]:
#         worksheet.write(f'A{i + 2}', i, cf)
#         worksheet.write(f'B{i + 2}', df_index.loc[i, 'Başlık'], cf)

writer.close()

if open_file:
    os.startfile(output_file_path)